# AgentCore Gateway — Elicitation + Sampling

## 개요

Elicitation을 사용하면 MCP 서버가 도구 호출 도중 실행을 일시 중지하고 클라이언트에 입력을 요청하거나(form-mode, URL-mode), 클라이언트가 재시도 전에 처리해야 하는 `URLElicitationRequiredError`를 표시할 수 있습니다. Sampling을 사용하면 서버가 `sampling/createMessage`를 통해 클라이언트의 LLM에 텍스트 생성을 위임할 수 있습니다. 다음과 같은 이유로 두 기능 모두 Gateway에서 **스트리밍과 세션을 활성화**해야 합니다.

- Elicitation은 `elicitation/create` 요청을 전달하기 위한 서버-클라이언트 SSE 채널과 응답을 연결하기 위한 세션이 필요합니다.
- Sampling은 `sampling/createMessage`에 동일한 SSE 채널을 사용합니다.

```bash
{
  "protocolConfiguration": {
    "mcp": {
      "sessionConfiguration": {
        "sessionTimeoutInSeconds": 3600
      },
      "streamingConfiguration": {
        "enableResponseStreaming": true
      }
    }
  }
}
```

이 Notebook에서는 `streamingConfiguration.enableResponseStreaming`과 `sessionConfiguration`을 **모두** 활성화한 Gateway를 구성하고(Lambda 인터셉터 없음), 전용 MCP 서버(`labelicitation`)를 대상으로 모든 Elicitation/Sampling 패턴을 엔드 투 엔드로 살펴봅니다.

![Elicitation 다이어그램 자리 표시자](./images/elicitation.png)

## 워크숍 로드맵

| 단계 | 수행할 작업 |
|---|---|
| **1** | Notebook을 설정합니다. |
| **2** | 스트리밍과 세션을 모두 활성화한 Gateway를 생성합니다. |
| **3** | `labelicitation` FastMCP 서버를 AgentCore Runtime에 배포합니다. |
| **4** | MCP 서버를 Gateway 대상에 연결합니다. |
| **5** | Form-mode Elicitation — `book_room`(단일 객체 스키마)을 실행합니다. |
| **6** | 불리언 확인 — `cancel_with_confirm`을 실행합니다. |
| **7** | 순차 Elicitation — `log_expense`(하나의 도구 호출에서 프롬프트 3개)를 실행합니다. |
| **8** | Sampling — `sampling_demo`(서버가 클라이언트의 LLM에 요청)를 실행합니다. |
| **9** | 장시간 연산 + Elicitation — `optimize_and_apply`를 실행합니다. |
| **10** | URL-mode Elicitation 흐름 §4.2 — `connect_external_account` + 완료 알림을 실행합니다. |
| **11** | 리소스를 정리합니다. |

## 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:---|:---|
| 튜토리얼 유형 | 대화형 |
| AgentCore 구성 요소 | AgentCore Gateway, AgentCore Identity, AgentCore Runtime |
| Gateway 대상 유형 | MCP 서버 |
| Gateway 기능 | 스트리밍 켜짐, 세션 켜짐, 인터셉터 없음 |
| MCP 전송 방식 | Streamable HTTP, 양방향 SSE |
| 인바운드 인증 | Cognito (M2M) |
| 아웃바운드 인증 | OAuth2 자격 증명 공급자를 통한 Cognito (M2M) |
| 사용 SDK | form-mode + sampling에는 boto3 + `mcp.client.session.ClientSession`, URL-mode에는 원시 httpx 사용(`mcp` 1.27.0은 아직 자동 처리하지 않음) |


### 1단계: 설정 및 사전 요구 사항

Jupyter(Python 3.10+ 커널), Node.js + npm, `us-west-2`의 AWS 자격 증명이 필요합니다. 아래에서 사용하는 form-mode + sampling 클라이언트를 실행하려면 로컬에 `mcp >= 1.27.0`이 필요합니다. 이전 버전은 서버의 `protocolVersion: 2025-11-25`를 거부합니다.

In [ ]:
# 현재 디렉터리의 requirements.txt 또는 pyproject.toml에서 설치
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
!npm install -g @aws/agentcore

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 이 Notebook에서 사용할 import 및 상수
import utils
import logging
import boto3
import json

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

REGION = boto3.Session().region_name
COGNITO_STACK_NAME = "agentcore-gateway-lab"
TEMPLATE_PATH = "cloudformation/cognito-signup-stack.yaml"
MCP_SERVER_NAME = "lab6elicitation"
GATEWAY_NAME = "ac-gateway-elicitation"

cfn = boto3.client("cloudformation", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)
print("REGION:", REGION)

### 2단계: Gateway 생성

### 2.1단계: CloudFormation으로 Cognito 배포

In [ ]:
outputs = utils.deploy_cognito_stack(cfn, COGNITO_STACK_NAME, TEMPLATE_PATH)

# Gateway 인바운드
gw_user_pool_id = outputs["UserPoolId"]
gw_client_id = outputs["GatewayClientId"]
gw_cognito_discovery_url = outputs["DiscoveryUrl"]
scopeString = outputs["GatewayScope"]
token_endpoint = outputs["TokenEndpoint"]
gw_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=gw_client_id
)["UserPoolClient"]["ClientSecret"]

# MCP 서버로의 아웃바운드(동일한 풀)
runtime_client_id = outputs["MCPClientId"]
runtime_cognito_discovery_url = gw_cognito_discovery_url
runtimeScopeString = outputs["MCPScope"]
runtime_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=runtime_client_id
)["UserPoolClient"]["ClientSecret"]

print(f"User Pool ID:       {gw_user_pool_id}")
print(f"Discovery URL:      {gw_cognito_discovery_url}")
print(f"Token endpoint:     {token_endpoint}")
print(f"Gateway client ID:  {gw_client_id}")
print(f"MCP client ID:      {runtime_client_id}")
print(f"Gateway scope:      {scopeString}")
print(f"MCP scope:          {runtimeScopeString}")

### 2.2단계: Gateway IAM 역할 생성

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role_with_region(
    GATEWAY_NAME, REGION
)
print("AgentCore Gateway role ARN:", agentcore_gateway_iam_role["Role"]["Arn"])

### 2.3단계: 스트리밍과 세션을 모두 활성화한 Gateway 생성

두 블록이 모두 포함되어 있습니다. Elicitation에는 두 블록이 모두 필요합니다. `interceptorConfigurations`는 **사용하지 않습니다**.

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [gw_client_id],
        "discoveryUrl": gw_cognito_discovery_url,
    }
}

create_response = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=agentcore_gateway_iam_role["Role"]["Arn"],
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {
            "supportedVersions": ["2025-11-25"],
            "streamingConfiguration": {"enableResponseStreaming": True},
            "sessionConfiguration": {"sessionTimeoutInSeconds": 3600},
        }
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="Elicitation gateway (streaming + sessions, no interceptor)",
)
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(f"Gateway ID:  {gatewayID}")
print(f"Gateway URL: {gatewayURL}")

### 3단계: AgentCore Runtime에 MCP 서버 배포

### 3.1단계: MCP 서버 코드 확인

`labelicitation`은 전체 Elicitation 기능을 제공합니다.

| 도구 | 시연 기능 |
|---|---|
| `book_room` | Form Elicitation — 단일 객체 스키마 |
| `cancel_with_confirm` | 불리언 확인 프롬프트 |
| `log_expense` | 하나의 도구 호출에서 세 번의 순차 Elicitation |
| `sampling_demo` | `sampling/createMessage` 왕복 처리 |
| `optimize_and_apply` | 장시간 연산 + Form Elicitation 게이트 |
| `connect_external_account` | URL-mode Elicitation 흐름 4.2 + `notifications/elicitation/complete` |
| `protected_resource` | URL Required Error 흐름 4.3(`-32042`) — 첫 번째 호출은 오류 발생, 두 번째 호출은 성공 |


In [ ]:
from IPython.display import Code

Code("mcpservers/app/labelicitation/main.py", language="python")

### 3.2단계: Agent 등록

In [ ]:
!cd mcpservers && agentcore add agent \
    --name {MCP_SERVER_NAME} \
    --type byo \
    --language Python \
    --protocol MCP \
    --code-location app/labelicitation \
    --authorizer-type CUSTOM_JWT \
    --discovery-url {runtime_cognito_discovery_url} \
    --allowed-clients {runtime_client_id} \
    --allowed-scopes {runtimeScopeString}

### 3.3단계: AgentCore CLI로 배포

In [ ]:
!cd mcpservers && agentcore deploy

In [ ]:
agent = utils.get_agent_status(MCP_SERVER_NAME)

mcp_arn = agent["identifier"]
mcp_url = agent["invocationUrl"]
mcp_id = mcp_arn.split("/")[-1]

print(f"mcp_arn: {mcp_arn}")
print(f"mcp_id:  {mcp_id}")
print(f"mcp_url: {mcp_url}")

### 4단계: MCP 서버를 Gateway 대상으로 연결

### 4.1단계: 아웃바운드 OAuth2 자격 증명 공급자

In [ ]:
identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name=f"{GATEWAY_NAME}-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {"discoveryUrl": runtime_cognito_discovery_url},
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### 4.2단계: Gateway 대상 생성

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### 4.3단계: 대상이 READY 상태인지 확인

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(json.dumps(list_targets_response, default=str, indent=2))

### 4.4단계: 인바운드 액세스 토큰 가져오기

In [ ]:
token_response = utils.get_token(
    token_endpoint, gw_client_id, gw_client_secret, scopeString
)
token = token_response["access_token"]
print("Token (truncated):", token[:60], "...")

### 4.5단계: form-mode + sampling 클라이언트 구성

`mcp.client.session.ClientSession`이 `elicitation/create` 및 `sampling/createMessage` 콜백을 처리합니다. Form Elicitation에서는 스키마를 적절한 기본값으로 채워 자동 수락하고, Sampling에서는 고정된 시뮬레이션 문자열을 반환합니다.


In [ ]:
from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(token_endpoint, gw_client_id, gw_client_secret, scopeString)[
        "access_token"
    ]

In [ ]:
# `mcp.initialize()`는 명세 핸드셰이크를 실행하고 Gateway에서 발급한
# `Mcp-Session-Id`를 캡처합니다. 이후 모든 `call_tool_streaming` 호출은
# 해당 헤더를 자동으로 다시 전송합니다.
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token)
init_info = mcp.initialize(
    client_info={"name": "elicit-demo", "version": "0.1"},
    capabilities={"elicitation": {}, "sampling": {}},
)
print(f"init HTTP {init_info['http_status']}, sid={mcp.session_id}")

## 5단계: Form-mode Elicitation — `book_room`

서버는 Pydantic 스키마를 통해 `room_type`, `nights`, `breakfast` 입력을 요청합니다. 콜백은 `room_type=single, nights=1, breakfast=True`로 자동 수락합니다. 서버는 예약 확인 문자열을 반환합니다.

In [ ]:
# book_room — `interactive_input_form`은 서버가 전송한 스키마
# (`room_type`, `nights`, `breakfast`)를 읽고 각 항목을 input()으로 입력받습니다.
outcome = mcp.call_tool_streaming(
    "mcp-server-target___book_room",
    {},
    elicitation_callback=utils.interactive_input_form,
    request_id=5,
)
utils.show("book_room", outcome)

## 6단계: 불리언 확인 — `cancel_with_confirm`

데이터를 변경하는 작업 전에 단일 필드 불리언 Elicitation을 수행합니다. 자동 수락은 `confirm=True`를 반환합니다.


In [ ]:
# cancel_with_confirm — 단일 불리언 Elicitation입니다. 서버가 데이터를 변경하는
# 작업의 확인을 요청하면 프롬프트가 실행됩니다.
outcome = mcp.call_tool_streaming(
    "mcp-server-target___cancel_with_confirm",
    {"order_id": 42},
    elicitation_callback=utils.interactive_input_form,
    request_id=6,
)
utils.show("cancel_with_confirm", outcome)

## 7단계: 순차 Elicitation — `log_expense`

하나의 도구 호출에서 category → description → confirm 순서로 세 개의 elicitation/create 요청을 처리합니다. 콜백은 각 요청을 차례로 채웁니다.


In [ ]:
# log_expense — 세 번의 순차 Elicitation(category → description →
# confirm)을 수행합니다. 서버가 전송하는 각 elicitation/create마다 콜백이 한 번씩
# 실행되므로 별도의 프롬프트 세 개가 차례로 표시됩니다.
outcome = mcp.call_tool_streaming(
    "mcp-server-target___log_expense",
    {"amount": 12.34},
    elicitation_callback=utils.interactive_input_form,
    request_id=7,
)
utils.show("log_expense", outcome)

## 8단계: Sampling — `sampling_demo`

sampling_demo — 서버가 `ctx.sample(prompt)`를 호출합니다. `bedrock_sampling` 콜백은 요청을 Amazon Bedrock(기본값은 Claude Haiku 4.5)으로 전달하고 모델의 응답을 반환합니다. 도구는 LLM이 생성한 내용을 그대로 반환합니다.


In [ ]:
outcome = mcp.call_tool_streaming(
    "mcp-server-target___sampling_demo",
    {"prompt": "hello"},
    sampling_callback=utils.bedrock_sampling,
    request_id=8,
)
utils.show("sampling_demo", outcome)

## 9단계: 장시간 연산 + Elicitation 게이트 — `optimize_and_apply`

`duration_seconds` 동안 연산하면서 `interval_seconds`마다 진행률 알림을 전송한 다음, Form Elicitation으로 사용자에게 데이터 변경 작업의 승인을 요청합니다. 장시간 연산을 수행하고 적용 전에 승인을 받기 위해 일시 중지해야 하는 AI 어시스턴트 워크로드를 재현합니다.


In [ ]:
# optimize_and_apply — 서버가 장시간 연산을 실행하면서 `interval_seconds`마다
# 진행률을 알린 다음 적용 여부를 묻습니다. progress_callback은 각 프레임이
# 도착할 때마다 출력하며, 연산이 끝나면 Elicitation 프롬프트가 한 번
# 실행됩니다.
outcome = mcp.call_tool_streaming(
    "mcp-server-target___optimize_and_apply",
    {"duration_seconds": 10, "interval_seconds": 2},
    elicitation_callback=utils.interactive_input_form,
    progress_callback=lambda p: print(
        f"  progress: {p.get('progress')}/{p.get('total')} {p.get('message', '')}"
    ),
    progress_token="optimize-apply",
    request_id=9,
)
utils.show("optimize_and_apply", outcome)

## 10단계: URL-mode Elicitation 흐름 §4.2 — `connect_external_account`

MCP 명세 2025-11-25에 따르면 URL-mode Elicitation에서는 서버가 JSON 양식 대신 클라이언트가 사용자에게 제시할 URL(OAuth 동의 화면, 결제 흐름 등)을 전송할 수 있습니다. 대역 외 상호 작용이 완료되면 서버가 `notifications/elicitation/complete`를 후속으로 전송할 수 있습니다.

`mcp.client.session.ClientSession` 1.27.0은 URL mode를 자동으로 처리하지 않으므로 원시 httpx를 사용해 SSE 처리를 직접 구현합니다. tools/call을 스트리밍하고, `elicitation/create` 중 `params.mode == "url"`인 요청을 확인한 뒤 `{action: "accept"}`를 POST로 반환하고 완료 알림을 확인합니다.

In [ ]:
def url_elicit_callback(params):
    """URL 모드 elicitation 콜백입니다. 서버가 URL과 elicitationId를 제공하면,
    실제 환경에서는 사용자 브라우저에서 URL을 열고(예: OAuth 동의)
    대역 외 흐름이 완료된 뒤 `accept`를 반환합니다. 여기서는 로그를 남기고
    즉시 수락합니다.
    """
    if params.get("mode") == "url":
        url = params.get("url")
        eid = params.get("elicitationId")
        print(f"  elicitation/create mode=url url={url!r} eid={eid!r}")
        print("  -> sending {action: accept}")
        return {"action": "accept"}
    return {"action": "decline"}


def on_notification(method, params):
    if method == "notifications/elicitation/complete":
        print(
            f"  notifications/elicitation/complete eid={params.get('elicitationId')!r}"
        )


outcome = mcp.call_tool_streaming(
    "mcp-server-target___connect_external_account",
    {"completion_delay_s": 1.0},
    elicitation_callback=url_elicit_callback,
    notification_callback=on_notification,
    request_id=100,
)
print()
print(f"final_result: {outcome.get('result')}")

## 9단계: 리소스 정리

아래 셀의 주석을 해제하면 이 Notebook에서 생성한 Gateway, OAuth2 자격 증명 공급자, MCP 서버 Runtime 및 IAM 역할을 삭제할 수 있습니다. Cognito CloudFormation 스택은 여러 실습에서 공유하므로 모든 실습을 마친 경우가 아니라면 그대로 두세요.

In [ ]:
# utils.delete_gateway(gateway_client, gatewayID)

In [ ]:
# identity_client.delete_oauth2_credential_provider(name=f"{GATEWAY_NAME}-identity")

In [ ]:
# !cd mcpservers && agentcore remove agent --name {MCP_SERVER_NAME} -y
# !cd mcpservers && agentcore deploy -y

In [ ]:
# # ## 다른 lab에서 이 stack을 사용하지 않을 때 Cognito stack 삭제
# print(f"Deleting stack {COGNITO_STACK_NAME}...")
# cfn.delete_stack(StackName=COGNITO_STACK_NAME)
# cfn.get_waiter("stack_delete_complete").wait(StackName=COGNITO_STACK_NAME)
# print(f"✅ Stack {COGNITO_STACK_NAME} deleted")

In [ ]:
# utils.delete_iam_role(f"agentcore-{GATEWAY_NAME}-role")